# 365 Probabilidades - Dia #115
## Qual a probabilidade de alguém devolver a sua carteira?

**Tipo:** Comportamental
**Data de publicação:** 2026-10-06
**Ferramenta:** Python
**Decisão analisada:** Vale a pena esperar a carteira voltar?
**Hashtag:** #365Probabilidades #Dia115

---

### 📖 A História

Entre 2013 e 2016, um grupo de pesquisadores entregou **17.303 carteiras** em balcões
de bancos, museus, correios, hotéis e repartições públicas, em **355 cidades de 40
países**.

Cada uma era uma caixinha transparente. Dentro, três cartões de visita com um e-mail,
uma lista de compras e uma chave. Quem entregava dizia que tinha achado na rua, pedia
para o funcionário resolver porque estava com pressa, e ia embora.

Depois disso, os pesquisadores só esperavam. O que eles mediram não foi a carteira
voltar para a mão do dono. Foi o funcionário escrever para o e-mail dos cartões dentro
de **cem dias**.

Metade das carteiras estava vazia. A outra metade tinha dinheiro dentro.

---

### 📚 O Conceito

A intuição é direta: quanto mais dinheiro dentro, maior a tentação, menor a chance de
alguém procurar o dono.

O desenho do estudo permite testar duas forças separadamente, e é aí que ele fica
interessante.

A **chave** vale alguma coisa para o dono e nada para quem acha. Em três países os
pesquisadores rodaram uma versão com dinheiro e sem chave. Se tirar a chave reduz o
contato, o que está agindo é a preocupação com o prejuízo do outro.

O **valor alto** testa a outra força. Se aumentar o dinheiro aumenta o contato, o que
está agindo é a dificuldade de se ver como alguém que roubou.

---

### 🧮 O Modelo

Cada taxa vira uma posterior de Jeffreys, **Beta(x + 0,5; n − x + 0,5)**, com intervalo
de credibilidade de 95%.

Comparações entre países são feitas na **diferença entre as posteriores**, e não por
sobreposição de intervalos individuais. Sobreposição de dois intervalos de 95% não é
teste de igualdade, e os dois números não dão o mesmo resultado neste dia.

O fator ×0,80 **não se aplica**. Não é survey autorrelatado. É comportamento observado
em campo, com desfecho registrado pelos pesquisadores.

**Fontes:**
- Cohn, Maréchal, Tannenbaum & Zünd (2019). *Civic honesty around the globe.*
  Science 365(6448), 70-73 — N=17.303 carteiras, 355 cidades, 40 países
- Dados individuais: Harvard Dataverse, doi:10.7910/DVN/YKBODN
- Previsões: amostra de 299 participantes, cenário dos Estados Unidos


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv
from collections import defaultdict

SEMENTE = 42
SORTEIOS = 200_000

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def mil(n):
    return f"{int(n):,}".replace(",", ".")

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- OS DADOS INDIVIDUAIS ---
# Cohn, Marechal, Tannenbaum & Zund (2019). Harvard Dataverse doi:10.7910/DVN/YKBODN.
# Uma linha por carteira entregue.

with open('dados115/carteiras.csv', encoding='utf-8') as f:
    carteiras = list(csv.DictReader(f))
with open('dados115/previsoes.csv', encoding='utf-8') as f:
    previsoes = list(csv.DictReader(f))

for c in carteiras:
    c['cond'] = int(c['cond'])
    c['response'] = int(c['response'])

# As quatro condicoes do experimento:
#   0 = sem dinheiro, com chave      -> desenho principal, 40 paises
#   1 = com US$ 13,45, com chave     -> desenho principal, 40 paises
#   2 = com US$ 94,15, com chave     -> extensao, 3 paises
#   3 = com US$ 13,45, SEM chave     -> extensao, 3 paises
ROTULO = {0: 'sem dinheiro', 1: 'com US$ 13,45', 2: 'com US$ 94,15',
          3: 'US$ 13,45 sem chave'}
PRINCIPAIS = (0, 1)
EXTENSAO = ('Poland', 'UK', 'USA')

PT = {'Switzerland':'Suíça','Denmark':'Dinamarca','Norway':'Noruega','Sweden':'Suécia',
      'Netherlands':'Holanda','New Zealand':'Nova Zelândia','Czech Republic':'República Tcheca',
      'Poland':'Polônia','Australia':'Austrália','Germany':'Alemanha','Croatia':'Croácia',
      'France':'França','Canada':'Canadá','Romania':'Romênia','Serbia':'Sérvia',
      'UK':'Reino Unido','Spain':'Espanha','Russia':'Rússia','USA':'Estados Unidos',
      'Argentina':'Argentina','Israel':'Israel','Greece':'Grécia','Portugal':'Portugal',
      'Italy':'Itália','Brazil':'Brasil','Chile':'Chile','South Africa':'África do Sul',
      'India':'Índia','Thailand':'Tailândia','Turkey':'Turquia','UAE':'Emirados Árabes',
      'Indonesia':'Indonésia','Mexico':'México','Ghana':'Gana','Malaysia':'Malásia',
      'Morocco':'Marrocos','Kazakhstan':'Cazaquistão','Kenya':'Quênia','China':'China',
      'Peru':'Peru'}

principal = [c for c in carteiras if c['cond'] in PRINCIPAIS]
paises = sorted({c['Country'] for c in principal})

print("=" * 70)
print("  O EXPERIMENTO")
print("=" * 70)
print(f"\n  Carteiras entregues:  {mil(len(carteiras))}")
print(f"  Países:               {len({c['Country'] for c in carteiras})}")
print(f"  Cidades:              {len({c['city'] for c in carteiras})}")
print(f"  Desfecho medido:      o funcionário escrever para o dono em até 100 dias\n")

print("  DESENHO PRINCIPAL · 40 países")
for k in PRINCIPAIS:
    sub = [c for c in carteiras if c['cond'] == k]
    print(f"    {ROTULO[k]:<22}{mil(len(sub)):>7} carteiras em "
          f"{len({c['Country'] for c in sub})} países")

print("\n  EXTENSÃO · só Estados Unidos, Reino Unido e Polônia")
for k in (2, 3):
    sub = [c for c in carteiras if c['cond'] == k]
    print(f"    {ROTULO[k]:<22}{mil(len(sub)):>7} carteiras em "
          f"{len({c['Country'] for c in sub})} países")
print("\n  As duas linhas de baixo NÃO valem para os 40 países.")
print("  Toda comparação global usa apenas o desenho principal.")

br = [c for c in principal if c['Country'] == 'Brazil']
print(f"\n  Brasil: {len(br)} carteiras em {len({c['city'] for c in br})} cidades.")
for k in PRINCIPAIS:
    print(f"    {ROTULO[k]:<22}{len([c for c in br if c['cond'] == k]):>7} carteiras")
print(f"  As cidades vêm anonimizadas por código no arquivo original.")
print(f"\n  ×0,80: não se aplica. Comportamento observado em campo.")
print("=" * 70)

In [ ]:
# --- O MODELO ---
# Todos os intervalos deste notebook sao de CREDIBILIDADE (quantis da posterior
# Beta), e nao intervalos de confianca frequentistas. Onde estiver escrito
# "IC 95%" por falta de espaco na tabela, leia "intervalo de credibilidade de 95%".
ger = np.random.default_rng(SEMENTE)

# Posterior de Jeffreys: Beta(x + 1/2, n - x + 1/2)
def posterior(sucessos, total):
    return ger.beta(sucessos + 0.5, total - sucessos + 0.5, SORTEIOS)

def de(lista):
    x = sum(c['response'] for c in lista)
    return posterior(x, len(lista)), x / len(lista) * 100, len(lista)

def bruta(lista):
    return sum(c['response'] for c in lista) / len(lista) * 100

def ic(amostra, pct=100):
    return np.percentile(amostra, 2.5) * pct, np.percentile(amostra, 97.5) * pct

def filtra(cond=None, pais=None, paises=None):
    s = principal if cond is None else [c for c in carteiras if c['cond'] == cond]
    if pais:   s = [c for c in s if c['Country'] == pais]
    if paises: s = [c for c in s if c['Country'] in paises]
    return s

# ══ CAMADA 1 · O ACHADO, NOS 40 PAÍSES ══
print("=" * 72)
print("  CAMADA 1 · O ACHADO · desenho principal, 40 países")
print("=" * 72)
print(f"\n  {'':24}{'contato':>10}{'IC 95%':>22}{'n':>10}")
glob = {}
for k in PRINCIPAIS:
    p, obs, n = de(filtra(cond=k))
    glob[k] = p
    lo, hi = ic(p)
    print(f"  {ROTULO[k]:<24}{vir(obs) + '%':>10}   [{vir(lo)}%; {vir(hi)}%]{mil(n):>10}")

efeito = (glob[1] - glob[0]) * 100
lo, hi = ic(efeito, 1)
print(f"\n  O dinheiro AUMENTA o contato: {vir(efeito.mean())} pontos "
      f"[{vir(lo)}; {vir(hi)}]")

# direcao pais a pais, e o teste formal das excecoes
por_pais = defaultdict(dict)
for c in principal:
    por_pais[c['Country']].setdefault(c['cond'], []).append(c['response'])

subiu, excecoes = [], []
for pais, d in por_pais.items():
    t0, t1 = np.mean(d[0]), np.mean(d[1])
    (subiu if t1 > t0 else excecoes).append(pais)
print(f"\n  Mesma direção em {len(subiu)} dos {len(por_pais)} países.")
print(f"  Exceções observadas: {', '.join(PT[p] for p in excecoes)}")
print(f"\n  Mas a queda nesses dois países é distinguível de zero?")
for pais in excecoes:
    p0, o0, n0 = de(filtra(cond=0, pais=pais))
    p1, o1, n1 = de(filtra(cond=1, pais=pais))
    d = (p1 - p0) * 100
    lo, hi = ic(d, 1)
    print(f"    {PT[pais]:<10}{vir(o0)}% → {vir(o1)}%   efeito {vir(d.mean()):>5} pp"
          f"   IC 95% [{vir(lo)}; {vir(hi)}]   P(negativo) = {vir((d < 0).mean() * 100, 0)}%")
print("    Nenhuma das duas. São quedas compatíveis com o acaso.")

# ══ CAMADA 2 · O MECANISMO ══
print("\n" + "=" * 72)
print("  CAMADA 2 · O MECANISMO · extensão em 3 países")
print("=" * 72)
print("\n  A chave vale para o dono e não vale para quem acha.")
print("  O valor alto testa a outra força.\n")
mec = {}
for k in (0, 3, 1, 2):
    p, obs, n = de(filtra(cond=k, paises=EXTENSAO))
    mec[k] = (p, obs, n)
    lo, hi = ic(p)
    print(f"  {ROTULO[k]:<24}{vir(obs) + '%':>10}   [{vir(lo)}%; {vir(hi)}%]{mil(n):>10}")

def compara(ka, kb, nome):
    d = (mec[ka][0] - mec[kb][0]) * 100
    lo, hi = ic(d, 1)
    print(f"  {nome:<38}{vir(d.mean()):>6} pp   [{vir(lo)}; {vir(hi)}]"
          f"   P(>0) = {vir((d > 0).mean() * 100)}%")
    return d

print()
d_chave  = compara(1, 3, "efeito de TIRAR A CHAVE, invertido")
d_valor  = compara(2, 1, "efeito de US$ 13,45 para US$ 94,15")
d_dinh   = compara(1, 0, "efeito do dinheiro, chave nas duas")
print("\n  Os autores leem o primeiro efeito como preocupação com o prejuízo do")
print("  dono, e o segundo como o custo de se enxergar como alguém que roubou.")
print("  Os números são do experimento. A leitura é interpretação deles.")

# ══ CAMADA 3 · O BRASIL ══
print("\n" + "=" * 72)
print("  CAMADA 3 · O BRASIL")
print("=" * 72)
brasil = {}
print()
for k in PRINCIPAIS:
    p, obs, n = de(filtra(cond=k, pais='Brazil'))
    brasil[k] = p
    lo, hi = ic(p)
    print(f"  {ROTULO[k]:<24}{vir(obs) + '%':>10}   [{vir(lo)}%; {vir(hi)}%]{n:>10}")

ef_br = (brasil[1] - brasil[0]) * 100
lo, hi = ic(ef_br, 1)
print(f"\n  Efeito do dinheiro no Brasil: {vir(ef_br.mean())} pontos [{vir(lo)}; {vir(hi)}]"
      f"   P(>0) = {vir((ef_br > 0).mean() * 100)}%")
print(f"  No conjunto dos 40 países:    {vir(efeito.mean())} pontos")

efeitos = sorted(((np.mean(d[1]) - np.mean(d[0])) * 100, p) for p, d in por_pais.items())[::-1]
pos_ef = [i for i, (_, p) in enumerate(efeitos, 1) if p == 'Brazil'][0]
print(f"\n  Entre os 40 países, o aumento brasileiro é o {pos_ef}º maior.")
print(f"  Terço de cima, mas com intervalo largo o bastante para se sobrepor")
print(f"  ao de quase todo mundo.")

p_br, o_br, n_brt = de(filtra(pais='Brazil'))
p_re, o_re, n_ret = de([c for c in principal if c['Country'] != 'Brazil'])
dif = (p_br - p_re) * 100
lo, hi = ic(dif, 1)
print(f"\n  Brasil contra o RESTANTE da amostra (as duas condições juntas):")
print(f"    Brasil {vir(o_br)}% (n = {n_brt})  ·  restante {vir(o_re)}% (n = {mil(n_ret)})")
print(f"    diferença {vir(dif.mean())} pontos   IC 95% [{vir(lo)}; {vir(hi)}]")
print(f"    P(taxa brasileira menor que a do restante) = {vir((dif < 0).mean() * 100)}%")
print(f"\n  O intervalo inclui zero e a probabilidade posterior é alta. Não é")
print(f"  contradição: são duas formas de escrever a mesma incerteza.")

# ══ CAMADA 4 · O RANKING E O QUE ELE NÃO SEPARA ══
print("\n" + "=" * 72)
print("  CAMADA 4 · O RANKING, E O QUANTO ELE SEPARA")
print("=" * 72)
print("\n  Ordenação construída aqui: as duas condições principais juntas,")
print("  países ordenados pela taxa observada de contato.\n")

amostras, ranking = {}, []
for pais in paises:
    p, obs, n = de(filtra(pais=pais))
    amostras[pais] = p
    lo, hi = ic(p)
    ranking.append(dict(pais=pais, taxa=obs, lo=lo, hi=hi, n=n))
ranking.sort(key=lambda r: -r['taxa'])
for i, r in enumerate(ranking, 1):
    r['pos'] = i
posicao = {r['pais']: r['pos'] for r in ranking}
rbr = next(r for r in ranking if r['pais'] == 'Brazil')

print(f"  Brasil: {rbr['pos']}º de {len(ranking)}, {vir(rbr['taxa'])}% "
      f"[{vir(rbr['lo'])}; {vir(rbr['hi'])}]")
print(f"  Extremos: {PT[ranking[0]['pais']]} {vir(ranking[0]['taxa'])}%  ·  "
      f"{PT[ranking[-1]['pais']]} {vir(ranking[-1]['taxa'])}%")

sobrepoe = [r for r in ranking if r['pais'] != 'Brazil'
            and r['lo'] < rbr['hi'] and r['hi'] > rbr['lo']]
print(f"\n  (a) Regra da sobreposição: países cujo intervalo individual de 95%")
print(f"      encosta no do Brasil: {len(sobrepoe)}")
print(f"      posições {min(r['pos'] for r in sobrepoe)} a "
      f"{max(r['pos'] for r in sobrepoe)}")

iguais = []
for pais in paises:
    if pais == 'Brazil':
        continue
    d = (amostras[pais] - amostras['Brazil']) * 100
    lo, hi = ic(d, 1)
    if lo < 0 < hi:
        iguais.append(pais)
iguais.sort(key=lambda p: posicao[p])
print(f"\n  (b) Comparação formal: posterior da diferença (país − Brasil),")
print(f"      países cujo intervalo de credibilidade de 95% dessa diferença")
print(f"      ainda contém zero: {len(iguais)}")
print(f"      {', '.join(PT[p] for p in iguais)}")
print(f"      posições {', '.join(str(posicao[p]) for p in iguais)}")
print(f"\n  (a) e (b) não medem a mesma coisa, e aqui não dão o mesmo número.")
print(f"  A regra da sobreposição classifica {len(sobrepoe)} países nesse grupo.")
print(f"  A comparação direta da diferença deixa {len(iguais)}.")
print(f"  Sobreposição de dois intervalos de 95% não é teste de igualdade.")

# ══ CAMADA 5 · O PALPITE ══
print("\n" + "=" * 72)
print(f"  CAMADA 5 · O QUE {len(previsoes)} PESSOAS ACHAVAM QUE IA ACONTECER")
print("=" * 72)
print("\n  As previsões foram coletadas para o cenário dos ESTADOS UNIDOS.")
print("  Portanto a comparação é com o que aconteceu nos Estados Unidos.\n")

prev = {i: np.array([float(p[f'prediction{i}']) for p in previsoes]) for i in (1, 2, 3)}
obs_eua = {}
print(f"  {'':24}{'previram':>11}{'aconteceu':>12}{'erro':>11}")
for i, k in [(1, 0), (2, 1), (3, 2)]:
    _, obs, n = de(filtra(cond=k, pais='USA'))
    obs_eua[k] = obs
    print(f"  {ROTULO[k]:<24}{vir(prev[i].mean()) + '%':>11}{vir(obs) + '%':>12}"
          f"{vir(prev[i].mean() - obs):>8} pp")

a, b, c_ = prev[1], prev[2], prev[3]
plana = (a == b) & (b == c_)
desce = (a >= b) & (b >= c_) & ~plana
sobe  = (a <= b) & (b <= c_) & ~plana
print(f"\n  Olhando o FORMATO da previsão de cada uma das {len(previsoes)} pessoas:")
print(f"    previram que o contato CAIRIA com mais dinheiro:  {desce.sum():>4} "
      f"({vir(desce.mean() * 100)}%)")
print(f"    previram que SUBIRIA, que foi o que aconteceu:    {sobe.sum():>4} "
      f"({vir(sobe.mean() * 100)}%)")
print(f"    previram linha plana:                             {plana.sum():>4} "
      f"({vir(plana.mean() * 100)}%)")
print(f"    nenhuma das direções:                             "
      f"{len(a) - desce.sum() - sobe.sum() - plana.sum():>4} "
      f"({vir((1 - desce.mean() - sobe.mean() - plana.mean()) * 100)}%)")
print(f"\n  Não é imprecisão. Quase dois terços apostaram no sentido contrário.")
print("=" * 72)

In [ ]:
# --- VISUALIZAÇÃO ---

# ══ GRÁFICO 1 · o achado, nos 40 países ══
fig1, ax1 = plt.subplots(figsize=(11, 6.5))
med = np.array([bruta(filtra(cond=k)) for k in PRINCIPAIS])
lo_ = np.array([np.percentile(glob[k], 2.5) * 100 for k in PRINCIPAIS])
hi_ = np.array([np.percentile(glob[k], 97.5) * 100 for k in PRINCIPAIS])
ax1.bar(range(2), med, color=[CINZA, DOURADO], alpha=0.9, width=0.45)
ax1.errorbar(range(2), med, yerr=[med - lo_, hi_ - med],
             fmt='none', ecolor='#333333', elinewidth=2, capsize=9)
for i, (v, h) in enumerate(zip(med, hi_)):
    ax1.text(i, h + 2.4, vir(v) + '%', ha='center', fontsize=21, fontweight='bold')
ax1.set_xticks(range(2))
ax1.set_xticklabels(['carteira vazia', 'com US$ 13,45 dentro'], fontsize=13)
ax1.set_ylabel('funcionário entrou em contato com o dono (%)', fontsize=11)
ax1.set_ylim(0, 74)
ax1.set_xlim(-0.55, 1.55)
ax1.annotate('', xy=(1, 62.5), xytext=(0, 62.5),
             arrowprops=dict(arrowstyle='<->', color=VERMELHO, lw=2))
ax1.text(0.5, 64.5, '+' + vir(efeito.mean()) + ' pontos', ha='center',
         fontsize=13.5, color=VERMELHO, fontweight='bold')
ax1.set_title('Mais dinheiro dentro, mais gente procurou o dono\n'
              '17.303 carteiras · 355 cidades · 40 países · 2013 a 2016',
              fontsize=14, pad=16)
plt.figtext(0.5, 0.005, 'Fonte: Cohn et al. (2019), Science · barras de erro = IC 95% '
            '· desfecho: contato por e-mail em até 100 dias  |  #365Probabilidades',
            ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=[0, 0.025, 1, 1])
plt.savefig('dia-115-grafico-01-achado.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · o mecanismo ══
fig2, ax2 = plt.subplots(figsize=(11.5, 6.8))
ordem = [0, 3, 1, 2]
m2 = np.array([mec[k][1] for k in ordem])
l2 = np.array([np.percentile(mec[k][0], 2.5) * 100 for k in ordem])
h2 = np.array([np.percentile(mec[k][0], 97.5) * 100 for k in ordem])
ax2.bar(range(4), m2, color=[CINZA, '#b9a77a', DOURADO, VERDE], alpha=0.9, width=0.5)
ax2.errorbar(range(4), m2, yerr=[m2 - l2, h2 - m2],
             fmt='none', ecolor='#333333', elinewidth=1.8, capsize=7)
for i, (v, h) in enumerate(zip(m2, h2)):
    ax2.text(i, h + 1.5, vir(v) + '%', ha='center', fontsize=16, fontweight='bold')
ax2.set_xticks(range(4))
ax2.set_xticklabels(['sem dinheiro\ncom chave', 'US$ 13,45\nSEM chave',
                     'US$ 13,45\ncom chave', 'US$ 94,15\ncom chave'], fontsize=11.5)
ax2.set_ylabel('entrou em contato com o dono (%)', fontsize=11)
ax2.set_ylim(0, 96)
ax2.annotate('', xy=(2, 84), xytext=(1, 84),
             arrowprops=dict(arrowstyle='<->', color=VERMELHO, lw=1.8))
ax2.text(1.5, 86, 'a chave: +' + vir(d_chave.mean()) + ' pp\nvale para o dono',
         ha='center', fontsize=10.5, color=VERMELHO, fontweight='bold')
ax2.annotate('', xy=(3, 84), xytext=(2, 84),
             arrowprops=dict(arrowstyle='<->', color=VERDE, lw=1.8))
ax2.text(2.5, 86, 'o valor: +' + vir(d_valor.mean()) + ' pp\nganho de quem acha',
         ha='center', fontsize=10.5, color=VERDE, fontweight='bold')
ax2.set_title('Duas manipulações para investigar o mecanismo\n'
              'extensão do estudo em Estados Unidos, Reino Unido e Polônia',
              fontsize=14, pad=16)
plt.figtext(0.5, 0.005, 'Fonte: Cohn et al. (2019), Science · a chave vale para o dono '
            'e não vale para quem acha  |  #365Probabilidades',
            ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=[0, 0.025, 1, 1])
plt.savefig('dia-115-grafico-02-mecanismo.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · o ranking e a incerteza ══
fig3, ax3 = plt.subplots(figsize=(9.5, 11))
ys = range(len(ranking))
for y, r in zip(ys, ranking):
    destaque = r['pais'] == 'Brazil'
    formal = r['pais'] in iguais
    cor = VERMELHO if destaque else (DOURADO if formal else '#bbbbbb')
    lw = 4.0 if destaque else (2.6 if formal else 1.6)
    ax3.plot([r['lo'], r['hi']], [y, y], color=cor, lw=lw,
             solid_capstyle='round', zorder=3 if destaque else 2)
    ax3.plot([r['taxa']], [y], 'o', color=cor, ms=6 if destaque else 3.6, zorder=4)
ax3.set_yticks(list(ys))
ax3.set_yticklabels([f"{r['pos']:>2}. {PT[r['pais']]}" for r in ranking], fontsize=9.5)
for t, r in zip(ax3.get_yticklabels(), ranking):
    if r['pais'] == 'Brazil':
        t.set_color(VERMELHO); t.set_fontweight('bold'); t.set_fontsize(11)
    elif r['pais'] in iguais:
        t.set_color('#8a7325')
ax3.invert_yaxis()
ax3.set_xlabel('entrou em contato com o dono (%) · intervalo de credibilidade de 95%',
               fontsize=11)
ax3.set_xlim(8, 92)
ax3.axvspan(rbr['lo'], rbr['hi'], color=VERMELHO, alpha=0.07, zorder=1)
ax3.set_title('O ranking parece preciso, e não é\n'
              f'em dourado, os {len(iguais)} países cujo intervalo de 95% da diferença '
              'para o Brasil ainda contém zero',
              fontsize=13.5, pad=16)
plt.figtext(0.5, 0.004, 'Fonte: Cohn et al. (2019), Science · as duas condições '
            'principais juntas · cerca de 400 carteiras por país  |  #365Probabilidades',
            ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=[0, 0.018, 1, 1])
plt.savefig('dia-115-grafico-03-ranking.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · previsão contra realidade, nos Estados Unidos ══
fig4, (ax4a, ax4b) = plt.subplots(1, 2, figsize=(13.5, 6.3),
                                  gridspec_kw={'width_ratios': [1.45, 1]})
xs = [0, 1, 2]
pv = [prev[i].mean() for i in (1, 2, 3)]
ob = [obs_eua[k] for k in (0, 1, 2)]
ax4a.plot(xs, pv, 'o-', color=CINZA, lw=2.6, ms=10, label='o que 299 pessoas previram')
ax4a.plot(xs, ob, 'o-', color=VERMELHO, lw=2.6, ms=10, label='o que aconteceu')
# o rotulo do ponto mais alto vai por cima, o do mais baixo por baixo
for x, vp, vo in zip(xs, pv, ob):
    alto, baixo = (vp, vo) if vp > vo else (vo, vp)
    cor_a = CINZA if vp > vo else VERMELHO
    cor_b = VERMELHO if vp > vo else CINZA
    ax4a.text(x, alto + 2.8, vir(alto) + '%', ha='center', fontsize=12,
              color=cor_a, fontweight='bold')
    ax4a.text(x, baixo - 5.4, vir(baixo) + '%', ha='center', fontsize=12,
              color=cor_b, fontweight='bold')
ax4a.set_xticks(xs)
ax4a.set_xticklabels(['carteira\nvazia', 'US$ 13,45', 'US$ 94,15'], fontsize=11.5)
ax4a.set_ylabel('entrou em contato com o dono (%)', fontsize=11)
ax4a.set_ylim(25, 88)
ax4a.set_xlim(-0.35, 2.35)
ax4a.legend(fontsize=10.5, loc='upper right', framealpha=0.95)
ax4a.set_title('Previsão e realidade, nos Estados Unidos', fontsize=13, pad=12)

cats = ['previram\nque CAIRIA', 'previram\nque SUBIRIA', 'linha\nplana', 'nenhuma\ndas duas']
outro = len(a) - desce.sum() - sobe.sum() - plana.sum()
vals = [desce.mean() * 100, sobe.mean() * 100, plana.mean() * 100,
        outro / len(a) * 100]
ax4b.bar(range(4), vals, color=[CINZA, VERMELHO, '#cccccc', '#cccccc'],
         alpha=0.9, width=0.62)
for i, v in enumerate(vals):
    ax4b.text(i, v + 1.6, vir(v) + '%', ha='center', fontsize=13, fontweight='bold')
ax4b.set_xticks(range(4))
ax4b.set_xticklabels(cats, fontsize=10)
ax4b.set_ylabel('das 299 pessoas (%)', fontsize=11)
ax4b.set_ylim(0, 76)
ax4b.set_title('O formato de cada previsão, uma a uma', fontsize=13, pad=12)
plt.figtext(0.5, 0.005, 'Fonte: Cohn et al. (2019), Science · as previsões foram '
            'coletadas para o cenário dos Estados Unidos  |  #365Probabilidades',
            ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=[0, 0.028, 1, 1])
plt.savefig('dia-115-grafico-04-previsao.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

No experimento brasileiro, a carteira era entregue no balcão de uma instituição.
Quando estava vazia, o dono foi contatado em **cerca de um terço** dos casos. Com
dinheiro dentro, em **quase metade**.

Leia de novo a segunda parte.

Quanto mais dinheiro, mais gente procurou o dono. Isso apareceu na mesma direção em
**38 dos 40 países**. México e Peru foram as duas exceções observadas, mas as quedas
nesses dois países não eram distinguíveis de zero.

E é o contrário do que as pessoas imaginam. Entre as 299 que responderam ao estudo de
previsão, **63,5% apostaram que a chance cairia** conforme o dinheiro aumentasse. Só
17,7% acertaram a direção. Não é imprecisão de nível. É o sentido trocado.

Duas manipulações do estudo ajudam a separar as explicações. Tirar a chave, que vale
para o dono e não vale para quem acha, reduziu o contato em **6,8 pontos**. Aumentar o
dinheiro de treze para noventa e quatro dólares elevou o contato em **10,8 pontos**.
Os autores leem o primeiro resultado como preocupação com o prejuízo causado ao outro,
e o segundo como o custo de se enxergar como alguém que roubou. Os números são do
experimento; a leitura é interpretação deles, e aponta para as duas coisas.

Num ranking dos quarenta países construído aqui, juntando as duas condições
principais, o Brasil aparece em **25º**.

Vinte e cinco parece uma informação precisa. Não é.

Quando olho a distribuição posterior da diferença entre o Brasil e cada um dos outros
países, **seis** deles têm intervalo de 95% dessa diferença ainda contendo zero. A regra
da sobreposição de intervalos individuais classificava dez países nesse grupo. As duas
regras não medem a mesma coisa. Em qualquer uma das duas, a ordem exata daquele trecho
da lista diz muito menos do que parece.

Nada disso mede o caráter de ninguém. O estudo mede uma situação muito específica: um
estranho entrega uma carteira encontrada a um funcionário de uma instituição e vai
embora. O que acontece depois diz alguma coisa sobre comportamento naquele contexto.
Não autoriza transformar a posição de um país numa nota de honestidade da sua população.

O 25º lugar é exato na planilha. O que ele significa fora dela é muito menos exato.

*Quantas vezes você já decidiu alguma coisa sobre alguém com uma diferença desse
tamanho?*

---

### ⚠️ Limitações do Modelo

- O desfecho não é a carteira voltar. É o funcionário escrever para o e-mail dos
  cartões em até cem dias. Carteira devolvida de outro jeito não entra na conta.
- As condições de US$ 94,15 e de "sem chave" só rodaram em Estados Unidos, Reino
  Unido e Polônia. Nenhuma conclusão sobre elas vale para os 40 países.
- As previsões foram coletadas para o cenário dos Estados Unidos e só podem ser
  comparadas com o que aconteceu nos Estados Unidos.
- Cerca de 400 carteiras por país, e metade disso por condição. É pouco para separar
  países vizinhos no ranking, e o notebook mostra isso de propósito.
- Todos os intervalos são de credibilidade (quantis da posterior Beta), e não
  intervalos de confiança frequentistas.
- Os dados são de 2013 a 2016, em balcões institucionais de contexto urbano. Não são
  uma estimativa direta do que acontece hoje, nem de uma carteira perdida na rua.
- As cidades brasileiras vêm anonimizadas por código, então não dá para olhar
  variação interna.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
